# DES Y6: exploring cosmic shear

This notebook computes the cosmic-shear predictions of the DES Y6 project with
`cosmolike_des_y6_interface`, the project's *compiled interface*: C and C++ code
built into a Python module (through the pybind11 library), imported below as
`ci`. Cosmic shear is the correlated distortion of galaxy shapes by weak
gravitational lensing, caused by the matter between those galaxies and the
observer. The notebook shows

- the shear angular power spectra $C_\ell^{EE}$ and the real-space correlation
  functions $\xi_\pm(\theta)$ of the four source-galaxy redshift bins;
- how $\Omega_m$, $n_s$, $A_s$, the angular binning, baryonic feedback and the
  numerical accuracy settings change them;
- the $\chi^2$ of a prediction in units of the forecast DES Y6 errors, against a
  synthetic data vector, for HMcode's feedback term and for the accuracy
  settings.

**Vocabulary.**

- A *tomographic bin* is one redshift slice of a galaxy sample. DES Y6 has four
  *source* bins, galaxies whose shapes measure the gravitational shear, and six
  *lens* bins, galaxies used as density tracers (they appear in
  EXAMPLE_EVALUATE1.ipynb). Four source bins give 10 distinct bin pairs (1,1),
  (1,2), …, (4,4). Redshifts come from broad-band colors (*photo-z*).
- The multipole $\ell$ is an angular frequency, roughly $180^\circ/\theta$;
  $C_\ell$ is the angular power spectrum at that frequency.
- The *data vector* is the ordered list of all two-point values. The DES Y6
  3×2pt vector has 1,300 entries: 26 logarithmic angular bins from 2.5 to 995.27
  arcmin for each of the 10 source-bin pairs of $\xi_+$ and of $\xi_-$ (entries
  0 to 519), then galaxy–galaxy lensing and clustering.
- A *mask* holds one 0/1 flag per data-vector entry. The DES Y6 scale cuts,
  `data/DESY6.mask`, keep 541 entries, 210 of them $\xi_\pm$ entries: 152 of
  $\xi_+$ and 58 of $\xi_-$.
- A *wrapper* is a Python function of this notebook that runs CAMB, hands the
  result and the nuisance parameters to `ci`, and returns NumPy arrays.

**The shipped data are placeholders.** `data/DESY6.dataset` names an all-ones
data vector, an identity covariance and `ones.mask`, which keeps all 1,300
entries, so a $\chi^2$ against them measures nothing. The $\chi^2$ sections
install three other inputs in their place: the project's forecast covariance,
the DES Y6 scale cuts and a synthetic data vector, a prediction of this
notebook.

**Conventions.**

- Angles $\theta$ are in arcminutes throughout.
- `ci` returns plain, dimensionless $C_\ell$, without the factor
  $\ell(\ell+1)/2\pi$. The shear-spectrum plotter multiplies by that factor in
  its absolute panels; the $\xi$ plotter draws $\theta\,\xi_\pm\times10^4$.
- Arrays count tomographic bins from 0, panel labels count from 1: the panel
  labeled $(1,2)$ draws the array slice `[:, 0, 1]`.

**Running.** Follow the [notebook steps of the project README](README.md#notebooks)
(activate Cocoa, set `OMP_NUM_THREADS`, start Jupyter) and run the cells in
order. Every wrapper call runs CAMB once, so a sweep of ten values costs ten
CAMB runs. The $\chi^2$ sections read `covariance/forecast_real.npz`, which
EXAMPLE_EVALUATE_COVARIANCE.ipynb or `covariance/compute_covariance.py` writes
with the optional covariance build
([computing covariances](README.md#computing_covariances)).

## Setup: packages, threads and figure style

The first cell imports the packages and sets the figure style; the second adds
CAMB and the shared package `cosmolike_notebook_utils` (imported as `cnu`) to
`sys.path`, the list of folders Python searches for modules, and prints which
copies were found.

- `OMP_NUM_THREADS` sets how many OpenMP threads (parallel workers inside the
  compiled code) CosmoLike uses. An OpenMP runtime reads it once, when it
  starts, so the README sets it in the shell before Jupyter starts; the cell
  sets it again before `ci` is imported. A library loaded in the same process
  can lower the count later, so every wrapper below passes `threads` to
  `ci.set_omp_threads` first, as the likelihood does before each evaluation.
- `text.usetex = True` typesets all figure text with LaTeX and needs a LaTeX
  installation; set it to `False` if the figures fail to render.
- `IniFile` (from GetDist) reads `key = value` text files such as the project's
  `.dataset` file.

In [ ]:
# Figures appear below each cell; 'retina' draws them at double resolution.
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import sys, platform, os
# OpenMP threads (parallel workers) of the compiled CosmoLike loops. The
# OpenMP runtime reads this variable once, when it starts, so it is set
# before ci is imported; the README sets it in the shell before Jupyter.
# Lower the value on a machine with fewer than eight cores.
os.environ['OMP_NUM_THREADS'] = '8'
import matplotlib
import math
from matplotlib import pyplot as plt
import numpy as np
# EuclidEmulator2: used only when non_linear_emul = 1
import euclidemu2
import scipy
# ci: the project's compiled CosmoLike module
import cosmolike_des_y6_interface as ci
# IniFile reads key = value text files such as the .dataset file
from getdist import IniFile
import itertools
import iminuit
import functools
print(sys.version)
print(os.getcwd())

# Every wrapper hands this count to ci.set_omp_threads before computing: a
# library loaded in the same process can lower the OpenMP team size
threads = int(os.environ['OMP_NUM_THREADS'])

# GENERAL PLOT OPTIONS
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'
matplotlib.rcParams['mathtext.rm'] = 'Bitstream Vera Sans'
matplotlib.rcParams['mathtext.it'] = 'Bitstream Vera Sans:italic'
matplotlib.rcParams['mathtext.bf'] = 'Bitstream Vera Sans:bold'
matplotlib.rcParams['xtick.bottom'] = True
matplotlib.rcParams['xtick.top'] = False
matplotlib.rcParams['ytick.right'] = False
matplotlib.rcParams['axes.edgecolor'] = 'black'
matplotlib.rcParams['axes.linewidth'] = '1.0'
matplotlib.rcParams['axes.labelsize'] = 'medium'
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.linewidth'] = '0.0'
matplotlib.rcParams['grid.alpha'] = '0.18'
matplotlib.rcParams['grid.color'] = 'lightgray'
matplotlib.rcParams['legend.labelspacing'] = 0.77
matplotlib.rcParams['savefig.bbox'] = 'tight'
matplotlib.rcParams['savefig.format'] = 'pdf'
# usetex typesets all figure text with LaTeX: it needs a LaTeX installation
matplotlib.rcParams['text.usetex'] = True

# Jupyter Notebook Display options
import IPython
IPython.display.display(IPython.display.HTML("<style>:root { --jp-notebook-max-width: 85% !important; }</style>"))
IPython.display.display(IPython.display.HTML("<style>div.output_scroll { height: 54em; }</style>"))

In [ ]:
# IMPORT CAMB
# This build folder exists only on Linux x86_64. Python skips a missing
# folder in sys.path, so camb may come from another entry; the print shows
# which copy was imported.
sys.path.insert(0, os.environ['ROOTDIR']+'/external_modules/code/CAMB/build/lib.linux-x86_64-'+os.environ['PYTHON_VERSION'])
import camb
from camb import model
print('Using CAMB %s installed at %s'%(camb.__version__,os.path.dirname(camb.__file__)))

# IMPORT SHARED NOTEBOOK UTILITIES (cosmolike_core)
# cnu: the CAMB helper and plotting functions shared by every project
sys.path.insert(0, os.environ['ROOTDIR']+'/external_modules/code/cosmolike_core')
import cosmolike_notebook_utils as cnu
print('Using cosmolike_notebook_utils installed at %s'%(os.path.dirname(cnu.__file__)))

## Run settings

These module-level variables (names defined at the top level of the notebook)
mirror keys of the likelihood YAML files, `likelihood/cosmic_shear.yaml` and
`EXAMPLE_EVALUATE2.yaml`:

| Variable | Meaning |
| --- | --- |
| `non_linear_emul` | Nonlinear matter power: 1 = EuclidEmulator2, 2 = the CAMB model named by `halofit_version`. |
| `halofit_version` | CAMB's nonlinear model: `"mead2020_feedback"`, HMcode 2020 with its baryonic-feedback term, as in the YAML; `"takahashi"`, the gravity-only Halofit fit of Takahashi et al. 2012. |
| `kmax`, `k_per_logint`, `CAMBAccuracyBoost` | CAMB's largest wavenumber in 1/Mpc, its samples per logarithmic $k$ interval and its accuracy setting, from the YAML's `theory: camb` block. |
| `CLprobe` | Probe combination of the $\chi^2$ sections; `"xi"` is cosmic shear alone. |
| `path`, `data_file` | The DES Y6 dataset file; `path` is relative to this notebook's folder. |
| `IA_model` | Intrinsic-alignment model: 0 = NLA, 1 = TATT. |
| `IA_redshift_evolution` | 3 = each IA amplitude follows a power law in $(1+z)/1.62$. |
| `IA_code` | 0 = the C implementation of FAST-PT, the algorithm for the perturbation-theory integrals of TATT. |
| `adopt_limber_gs`, `adopt_limber_gg`, `nonlimber_accuracyboost` | Projection choices of galaxy–galaxy lensing and clustering; the likelihood sets them for every probe selection, and they do not change cosmic shear. |
| `photoz_interpolation_type`, `photoz_zmid_convention` | How the $n(z)$ tables are interpolated, and whether their redshift column holds left bin edges (the DES Y6 files) or bin centers. |
| `internal_accuracyboost` | FAST-PT internal grid, as a fraction of its output table. |
| `integration_accuracy`, `lmax` | Quadrature level of the line-of-sight integrals, and the largest multipole of the $C_\ell$ tables behind the real-space transforms. |
| `ntheta`, `theta_min_arcmin`, `theta_max_arcmin` | 26 logarithmic angular bins from 2.5 to 995.27 arcmin, as in the dataset. |

*Intrinsic alignment* (IA) is the correlation of galaxy shapes with the local
tidal field; it adds to the lensing signal and must be modeled. NLA, the
nonlinear linear-alignment model of the DES Y6 YAML files, has one amplitude
$A_1$; TATT (tidal alignment and tidal torquing) adds a quadratic amplitude $A_2$
and $b_{TA}$, which weights the alignment by the source-galaxy density.

**HMcode feedback.** `EXAMPLE_EVALUATE2.yaml` also sets `HMCode_logT_AGN: 7.7`,
the heating temperature of the AGN feedback in HMcode 2020.
`cnu.get_camb_cosmology` passes only the model name to CAMB, so this notebook's
HMcode runs at CAMB's default, $\log_{10}(T_{\rm AGN}/{\rm K})=7.8$. The
baryonic-feedback section passes `halofit_version = "takahashi"`: a feedback
ratio from a simulation multiplies a gravity-only spectrum, and on top of
HMcode's feedback term it would count feedback twice (see
[baryonic feedback](README.md#baryons) in the README).

In [ ]:
# Nonlinear power: 1 = EuclidEmulator2, 2 = the CAMB model named below
non_linear_emul = 2
# HMcode 2020 with its feedback term, as in EXAMPLE_EVALUATE2.yaml. The
# YAML's HMCode_logT_AGN = 7.7 does not reach CAMB through
# cnu.get_camb_cosmology, which leaves CAMB's default log10(T_AGN/K) = 7.8.
# The baryonic-feedback section passes "takahashi", the gravity-only Halofit.
halofit_version = "mead2020_feedback"
# CAMB inputs of the YAML's theory block: kmax in 1/Mpc, samples per
# logarithmic k interval and CAMB's AccuracyBoost. cnu.get_camb_cosmology
# raises kmax and k_per_logint further with the CAMB accuracy.
kmax = 5.0
k_per_logint = 10
CAMBAccuracyBoost = 1.05
# Probe combination of the chi^2 sections: cosmic shear alone
CLprobe = "xi"

# Relative to this notebook's folder; start_cocoa.sh links
# external_modules/data/des_y6 to this project's data folder.
path = "../../external_modules/data/des_y6"
data_file = "DESY6.dataset"

# IA_model: 0 = NLA, 1 = TATT. IA_redshift_evolution = 3: power law in
# (1+z)/1.62, so A1 = [amplitude, exponent, ...] and likewise A2.
IA_model = 0
IA_redshift_evolution = 3
IA_code = 0  # 0 = C FASTPT, 1 = Python FAST-PT (NLA always uses 0)

# Projection of galaxy-galaxy lensing and clustering (1 = Limber, 0 = exact
# below ell = 150) and the distance grid of the exact projection: set as
# in the likelihood, without effect on cosmic shear
adopt_limber_gs = 1
adopt_limber_gg = 0
nonlimber_accuracyboost = 1.0

# n(z) photo-z conventions (mirror the likelihood yaml keys):
# interpolation 0 = cspline, 1 = linear, 2+ = Steffen monotone;
# z column 0 = Z_LOW (left bin edges, the DES Y6 files), 1 = Z_MID
photoz_interpolation_type = 0
photoz_zmid_convention = 0

# C FAST-PT grid of the TATT and nonlinear-bias tables, as a fraction
# of its output table (mirrors the likelihood yaml key
# internal_accuracyboost, 1.0 in likelihood/*.yaml; the C default,
# used when nothing sets it, is 0.5)
internal_accuracyboost = 1.0

# Quadrature level 0 (the coarsest Gauss-Legendre rules) and the largest
# multipole of the C_ell tables behind the real-space transforms
integration_accuracy = 0
lmax = 50000

# 26 logarithmic angular bins from 2.5 to 995.27 arcmin, as in the dataset
ntheta = 26
theta_min_arcmin = 2.5
theta_max_arcmin = 995.2679263837432

## Fiducial parameters

The default values of the wrapper arguments are the `sampler: evaluate:
override` block of `EXAMPLE_EVALUATE2.yaml`; the parameters that block does not
list keep their fixed values of `likelihood/params_source.yaml`. Cosmic shear
needs only the source-galaxy nuisance parameters:

- Cosmology: `As_1e9` $=10^9A_s$, the primordial amplitude; `ns` $=n_s$; `H0` in
  km/s/Mpc; `omegab` $=\Omega_b$; `omegam` $=\Omega_m$, all matter including
  massive neutrinos; `mnu` $=\sum m_\nu$ in eV; `w` $=w_0$ and `w0pwa` $=w_0+w_a$,
  so $w_a=0$ here.
- Source photo-z shifts `DES_DZ_S1..4`: each source $n(z)$ is shifted in
  redshift by $\Delta z_i$.
- Shear calibration `DES_M1..4`: the measured shear of bin $i$ is $1+m_i$ times
  the true shear.
- NLA with power-law evolution: `DES_A1_1` is $A_1$ at $1+z=1.62$ and `DES_A1_2`
  its exponent $\eta_1$. The YAML flags its value, $A_1=0.769=1/1.3$, for a check
  of the sign and pivot conventions. The TATT terms `DES_A2_1`, `DES_A2_2` and
  `DES_BTA_1` are fixed to zero.

In [ ]:
# Cosmology: As_1e9 = 10^9 A_s, H0 in km/s/Mpc, mnu = sum of neutrino
# masses in eV; w0pwa = w0 + wa, so wa = w0pwa - w = 0
As_1e9 = 2.1
ns = 0.96
H0 = 67.0
omegab = 0.049
omegam = 0.319
mnu = 0.06
w = -1.0
w0pwa = -1.0
# Source bins 1-4: photo-z shifts Delta z_i and shear calibration m_i
DES_DZ_S1 = 0.0
DES_DZ_S2 = 0.0
DES_DZ_S3 = 0.0
DES_DZ_S4 = 0.0
DES_M1 = -0.00339767
DES_M2 = 0.0064552
DES_M3 = 0.01593935
DES_M4 = 0.00169811
# NLA: (A1, eta1) = (DES_A1_1, DES_A1_2). The TATT terms (A2, eta2) =
# (DES_A2_1, DES_A2_2) and b_TA = DES_BTA_1 are fixed to zero in
# likelihood/params_source.yaml; NLA reads none of them.
DES_A1_1 = 0.7692307692307692
DES_A1_2 = -1.0
DES_A2_1 = 0.0
DES_A2_2 = 0.0
DES_BTA_1 = 0.0

## Stage 1, CAMB: matter power spectrum, growth and distances

Each wrapper starts with one CAMB run through `cnu.get_camb_cosmology`, shared by
every Cocoa project; `help(cnu.get_camb_cosmology)` prints its documentation.
It returns the ten arrays that `ci.set_cosmology` reads, in CosmoLike's units:

| Returned names | Content |
| --- | --- |
| `log10k_interp_2D`, `z_interp_2D` | The grids of $\log_{10}k$, with $k$ in $h$/Mpc, and of redshift of the power-spectrum tables. |
| `lnPL`, `lnPNL` | $\ln P(k,z)$, linear and nonlinear, with $P$ in (Mpc/$h$)$^3$, each table flattened into one array. |
| `G_growth`, `z_growth` | $G(z)=D(z)(1+z)$, with $D$ the linear growth factor, normalized at the last redshift of the 2D grid, on its own redshift grid. |
| `z_interp_1D`, `chi` | The comoving distance $\chi(z)$ in Mpc/$h$. |
| `omegan2`, `lnPL_cb` | $\Omega_\nu h^2$ and the linear power of cold dark matter plus baryons, read by halo-model quantities; the wrappers pass both, as the likelihood does. |

*Flattened* means the 2D table $(z,k)$ is stored as one 1D array, column after
column (Fortran order), the memory layout the compiled code expects.

Two knobs reach CAMB: it runs with accuracy
`CAMBAccuracyBoost`$\times(1+(\texttt{AccuracyBoost}-1)/3)$, and the redshift and
wavenumber grids grow with `CLAccuracyBoost`$\times$`AccuracyBoost`. The helper
also raises `kmax` and `k_per_logint` with the CAMB accuracy: at the YAML's
`CAMBAccuracyBoost = 1.05`, CAMB samples up to $k=5.75$/Mpc with 11 points per
logarithmic interval.

**Comparison with `EXAMPLE_EVALUATE2.yaml`.** A Cobaya run of the YAML gives
slightly different spectra: the helper's CAMB has one massive neutrino with
$N_{\rm eff}=3.046$, where the YAML has three massive states and
$N_{\rm eff}=3.044$; HMcode's AGN temperature differs as the run settings
describe; and the CAMB $k$ sampling grows with the accuracy setting.

In [ ]:
# get_camb_cosmology is shared by every project's notebooks; call it as
# cnu.get_camb_cosmology (imported above). The CAMB side runs with
# CAMBAccuracyBoost*(1 + (AccuracyBoost - 1)/3); kmax, k_per_logint, the
# chi(z) node count and lens_potential_accuracy grow with that CAMB boost;
# the EuclidEmulator2 boost (non_linear_emul = 1) applies only at z < 10.
# help(cnu.get_camb_cosmology) prints the full documentation.

## Stages 2 to 5: the harmonic-space wrapper `C_ss_tomo_limber`

After CAMB, each call runs four more stages:

2. **Accuracy of the C code.** `ci.set_omp_threads` restores the OpenMP team
   size. `init_ntable_lmax` sets the largest multipole of the spectra tabulated
   for real-space transforms (`lmax` = 50,000 at boost 1, plus 20,000 per unit
   of boost above 1); `init_accuracy_boost` sets the size of CosmoLike's
   interpolation tables and the *quadrature level*, which picks how many points
   evaluate each line-of-sight integral; `init_photoz_conventions` sets how the
   $n(z)$ tables are read. The level passed is
   `CLIntegrationAccuracy` $+\,|3(\text{boost}-1)|$, rounded down, with boost
   $=$ `CLAccuracyBoost`$\times$`AccuracyBoost`: any boost different from 1,
   above or below, raises it.
3. **`ci.set_cosmology`** copies the CAMB tables into CosmoLike, which reads them
   in every later calculation.
4. **Nuisance setters.** `set_nuisance_shear_photoz` receives the four
   $\Delta z_i$ and `set_nuisance_ia` the IA lists. With
   `IA_redshift_evolution = 3` only the first two entries of `A1` and of `A2`
   (amplitude, exponent) and the first entry of `B_TA` are read; the trailing
   zeros make each list as long as the number of source bins, which the setter
   requires. `reset_bary_struct` switches tabulated baryonic feedback off;
   `init_baryons_contamination` switches it on (see the baryon section), and
   the wrapper refuses it together with HMcode's feedback model.
5. **The data-vector call** `ci.C_ss_tomo_limber(l=ell)` returns a pair
   `(EE, BB)` of arrays with shape `[n_ell, n_source, n_source]`. Only the
   entries `[:, i, j]` with $i\le j$ are filled; the others are zero. The values
   are plain, dimensionless $C_\ell$ in the *Limber approximation*, which
   evaluates the matter power at $k=(\ell+1/2)/\chi$ along the line of sight
   instead of integrating over all wavenumbers; its error shrinks as $\ell$ grows
   and as the radial weight of a bin widens, and lensing weights are broad in
   redshift.

The E mode is the curl-free pattern of the shear field, the pattern lensing
produces. In this model a B mode (curl pattern) comes only from the TATT terms:
NLA aligns galaxy shapes with the tidal field in an E-mode pattern, so with NLA
the `BB` array is zero. The shear calibration $(1+m_i)(1+m_j)$ enters only the
assembled data vector (`get_datavector` in the $\chi^2$ section), never the
spectra or the correlation functions, so the wrappers do not take it.

**Python detail.** The default values in the `def` line (`omegam = omegam`, ...)
are copied once, when the cell runs: rerun the cell after editing the fiducial
values or the run settings. Names that are not arguments (`mnu`,
`non_linear_emul`, `lmax`, `threads`, the photo-z conventions and `ini`) are read
from the notebook at every call; `ini` is created by the initialization cell
further below, which must run before the first call.

In [ ]:
def C_ss_tomo_limber(ell,
                     omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     ns = ns,
                     As_1e9 = As_1e9,
                     w = w,
                     w0pwa = w0pwa,
                     A1 = [DES_A1_1, DES_A1_2, 0, 0],
                     A2 = [DES_A2_1, DES_A2_2, 0, 0],
                     BTA = [DES_BTA_1, 0, 0, 0],
                     shear_photoz_bias = [DES_DZ_S1, DES_DZ_S2, DES_DZ_S3, DES_DZ_S4],
                     baryon_sims = None,
                     halofit_version = halofit_version,
                     AccuracyBoost = 1.0,
                     kmax = kmax,
                     k_per_logint = k_per_logint,
                     CAMBAccuracyBoost = CAMBAccuracyBoost,
                     CLAccuracyBoost = 1.0,
                     CLIntegrationAccuracy = integration_accuracy):

    # A simulation's power ratio multiplies a gravity-only spectrum; on top
    # of HMcode's feedback term it would count feedback twice. Checked
    # before CAMB runs, so a refused call changes nothing.
    if baryon_sims is not None and halofit_version == "mead2020_feedback":
        raise ValueError(
            f"baryon_sims = {baryon_sims} with halofit_version = "
            f"{halofit_version}: the simulation ratio needs a gravity-only "
            "spectrum; pass halofit_version = 'takahashi'")

    # Stage 1, CAMB: the ten arrays of the CAMB section
    (log10k_interp_2D, z_interp_2D, lnPL, lnPNL, G_growth, z_growth,
     z_interp_1D, chi, omegan2, lnPL_cb) = cnu.get_camb_cosmology(
        omegam = omegam, omegab = omegab, H0 = H0, ns = ns,
        As_1e9 = As_1e9, w = w, w0pwa = w0pwa, mnu = mnu,
        non_linear_emul = non_linear_emul,
        halofit_version = halofit_version,
        AccuracyBoost = AccuracyBoost, kmax = kmax,
        k_per_logint = k_per_logint,
        CAMBAccuracyBoost = CAMBAccuracyBoost,
        CLAccuracyBoost = CLAccuracyBoost)

    # Stage 2, accuracy of the C code. The OpenMP team size comes first: a
    # library loaded in this process can lower it. The quadrature level
    # grows by 3 per unit of |boost - 1|, so a boost below 1 raises it too.
    ci.set_omp_threads(n = threads)
    CLAccuracyBoost = CLAccuracyBoost * AccuracyBoost
    CLIntegrationAccuracy = max(0, CLIntegrationAccuracy + abs(3*(CLAccuracyBoost-1.0)))
    ci.init_ntable_lmax(lmax = int(lmax + 20000*(CLAccuracyBoost-1)))
    ci.init_accuracy_boost(accuracy_boost = CLAccuracyBoost,
                           integration_accuracy = int(CLIntegrationAccuracy))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))

    # Stage 3: copy the CAMB tables into CosmoLike, with omegab, omegan2 and
    # the cold-dark-matter-plus-baryon power lnPL_cb as the likelihood
    # sends them
    ci.set_cosmology(omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     log10k_2D = log10k_interp_2D,
                     z_2D = z_interp_2D,
                     lnP_linear = lnPL,
                     lnP_linear_cb = lnPL_cb,
                     lnP_nonlinear = lnPNL,
                     G = G_growth,
                     z_G = z_growth,
                     z_1D = z_interp_1D,
                     chi = chi,
                     omegan2 = omegan2)

    # Stage 4, nuisance parameters. With IA_redshift_evolution = 3 the IA
    # setter reads A1[0:2], A2[0:2] and BTA[0]; the zeros pad each list to
    # the number of source bins, which the setter requires.
    ci.set_nuisance_shear_photoz(bias = shear_photoz_bias)
    ci.set_nuisance_ia(A1 = A1, A2 = A2, B_TA = BTA)

    # Baryonic feedback off (gravity-only power), or the tabulated power
    # ratio of the named simulation; ini comes from the initialization cell
    if baryon_sims is None:
        ci.reset_bary_struct()
    else:
        ci.init_baryons_contamination(sim = baryon_sims,
            allsims = ini.relativeFileName('all_sims_hdf5_file'))

    # Stage 5: the pair (EE, BB), each [n_ell, n_source, n_source] with only
    # [:, i, j], i <= j, filled: plain dimensionless Limber C_ell
    return ci.C_ss_tomo_limber(l = ell)

## The real-space wrapper `xi`

For galaxy pairs separated by $\theta$, split each galaxy's shear into a
tangential and a cross component relative to the line joining the pair. The
correlation functions are
$\xi_\pm(\theta)=\langle\gamma_t\gamma_t\rangle\pm\langle\gamma_\times\gamma_\times\rangle$.

`xi(...)` runs the same stages and adds `ci.init_binning(ntheta, theta_min,
theta_max)`, which defines `ntheta` logarithmic angular bins between the two
angles in arcmin. `ci.xi_pm_tomo()` transforms the tabulated $C_\ell$, up to the
`init_ntable_lmax` multipole, into $\xi_+$, a weighted sum of
$C_\ell^{EE}+C_\ell^{BB}$, and $\xi_-$, a weighted sum of $C_\ell^{EE}-C_\ell^{BB}$,
with Legendre-polynomial weights averaged over each angular bin rather than
evaluated at its center.

The wrapper returns the tuple `(theta, xi_plus, xi_minus)`: `theta` holds the
area-weighted centers of the angular bins in arcmin,
$\theta=\frac{2}{3}\,\frac{\theta_{\max}^3-\theta_{\min}^3}{\theta_{\max}^2-\theta_{\min}^2}$
(`ci.get_binning_real_space()`), and each $\xi$ array has the shape
`[n_theta, n_source, n_source]`, symmetric in the two bin indices. This tuple is
the input format of `cnu.plot_xi`.

In [ ]:
def xi(ntheta = ntheta,
       theta_min_arcmin = theta_min_arcmin,
       theta_max_arcmin = theta_max_arcmin,
       omegam = omegam,
       omegab = omegab,
       H0 = H0,
       ns = ns,
       As_1e9 = As_1e9,
       w = w,
       w0pwa = w0pwa,
       A1 = [DES_A1_1, DES_A1_2, 0, 0],
       A2 = [DES_A2_1, DES_A2_2, 0, 0],
       BTA = [DES_BTA_1, 0, 0, 0],
       shear_photoz_bias = [DES_DZ_S1, DES_DZ_S2, DES_DZ_S3, DES_DZ_S4],
       baryon_sims = None,
       halofit_version = halofit_version,
       AccuracyBoost = 1.0,
       kmax = kmax,
       k_per_logint = k_per_logint,
       CAMBAccuracyBoost = CAMBAccuracyBoost,
       CLAccuracyBoost = 1.0,
       CLIntegrationAccuracy = integration_accuracy):

    # A simulation's power ratio multiplies a gravity-only spectrum
    if baryon_sims is not None and halofit_version == "mead2020_feedback":
        raise ValueError(
            f"baryon_sims = {baryon_sims} with halofit_version = "
            f"{halofit_version}: the simulation ratio needs a gravity-only "
            "spectrum; pass halofit_version = 'takahashi'")

    # Stage 1, CAMB
    (log10k_interp_2D, z_interp_2D, lnPL, lnPNL, G_growth, z_growth,
     z_interp_1D, chi, omegan2, lnPL_cb) = cnu.get_camb_cosmology(
        omegam = omegam, omegab = omegab, H0 = H0, ns = ns,
        As_1e9 = As_1e9, w = w, w0pwa = w0pwa, mnu = mnu,
        non_linear_emul = non_linear_emul,
        halofit_version = halofit_version,
        AccuracyBoost = AccuracyBoost, kmax = kmax,
        k_per_logint = k_per_logint,
        CAMBAccuracyBoost = CAMBAccuracyBoost,
        CLAccuracyBoost = CLAccuracyBoost)

    # Stage 2, accuracy of the C code, as in C_ss_tomo_limber
    ci.set_omp_threads(n = threads)
    CLAccuracyBoost = CLAccuracyBoost * AccuracyBoost
    CLIntegrationAccuracy = max(0, CLIntegrationAccuracy + abs(3*(CLAccuracyBoost-1.0)))
    ci.init_ntable_lmax(lmax = int(lmax + 20000*(CLAccuracyBoost-1)))
    ci.init_accuracy_boost(accuracy_boost = CLAccuracyBoost,
                           integration_accuracy = int(CLIntegrationAccuracy))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))
    # ntheta logarithmic bins in arcmin (int: the binding accepts only a
    # Python int). A new binning gets a new cache key: kernels are rebuilt.
    ci.init_binning(ntheta_bins = int(ntheta),
                    theta_min_arcmin = theta_min_arcmin,
                    theta_max_arcmin = theta_max_arcmin)

    # Stage 3: copy the CAMB tables into CosmoLike
    ci.set_cosmology(omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     log10k_2D = log10k_interp_2D,
                     z_2D = z_interp_2D,
                     lnP_linear = lnPL,
                     lnP_linear_cb = lnPL_cb,
                     lnP_nonlinear = lnPNL,
                     G = G_growth,
                     z_G = z_growth,
                     z_1D = z_interp_1D,
                     chi = chi,
                     omegan2 = omegan2)

    # Stage 4: nuisance parameters and baryons, as in C_ss_tomo_limber
    ci.set_nuisance_shear_photoz(bias = shear_photoz_bias)
    ci.set_nuisance_ia(A1 = A1, A2 = A2, B_TA = BTA)

    if baryon_sims is None:
        ci.reset_bary_struct()
    else:
        ci.init_baryons_contamination(sim = baryon_sims,
            allsims = ini.relativeFileName('all_sims_hdf5_file'))

    # Stage 5: bin-averaged xi_+ and xi_-, each [n_theta, n_source, n_source]
    (xip, xim) = ci.xi_pm_tomo()
    # Bin centers in arcmin first: the tuple cnu.plot_xi expects
    return (ci.get_binning_real_space(), xip, xim)

## The shared plotting functions

`cnu.plot_C_ss_tomo_limber` and `cnu.plot_xi` draw one panel per source-bin pair
$(i,j)$ in a lower triangle, in one of two modes:

- **Absolute**, without a reference argument: each panel has its own y range;
  the spectra appear as $\ell(\ell+1)C_\ell/2\pi$ on a logarithmic axis, the
  correlation functions as $\theta\,\xi_\pm\times10^4$ on a linear axis.
  `ylim = [a, b]` multiplies each panel's minimum by $a$ and its maximum by $b$.
- **Reference**, with `C_ss_ref=` or `xi_ref=`: each panel shows the fractional
  difference value/reference $-\,1$ on one shared linear band. `ylim` is then
  written around 1: `ylim = (0.96, 1.04)` draws the band from $-0.04$ to $+0.04$.

A list `param` of swept values colors the curves and the colorbar; `legend` names
the curves instead. `thetashow` sets the horizontal range of `plot_xi` in arcmin;
its default, 3 to 250 arcmin, would hide the DES Y6 bins above 250 arcmin, so
every call below passes the full range. `help(cnu.plot_xi)` lists every option.

In [ ]:
# plot_C_ss_tomo_limber is shared by every project; call it as
# cnu.plot_C_ss_tomo_limber (imported above). help(cnu.plot_C_ss_tomo_limber)
# prints every argument; the markdown above explains the two modes.

In [ ]:
# plot_xi is shared by every project; call it as cnu.plot_xi (imported
# above). help(cnu.plot_xi) prints every argument; the markdown above
# explains the two modes.

## Initialize CosmoLike once

The next cell configures the compiled module before the first wrapper call, in
the order of the likelihood's `initialize`
(`likelihood/_cosmolike_prototype_base.py`); the wrappers then change only
cosmology, nuisance parameters and accuracy.

- `ini` reads `DESY6.dataset`; `relativeFileName(key)` returns the file named
  under `key`, with the dataset's folder prepended.
- `initial_setup` resets every CosmoLike setting to its default.
- `init_fpt_internal_boost` comes before the first `init_accuracy_boost` call,
  because that call scales the value it finds then.
- `init_nonlimber_accuracy_boost`, `init_adopt_limber_gs`, `init_adopt_limber_gg`,
  `init_include_HOD_GX(0)` and `init_include_halo_IA(0)` repeat the likelihood's
  choices for the lens probes; they leave cosmic shear unchanged.
- `init_cosmo_runmode(is_linear=False)` selects the nonlinear matter power.
- `init_redshift_distributions_from_files` reads the lens and source $n(z)$
  tables: a redshift column, then one column per tomographic bin.
- `init_IA` selects the IA model, its redshift evolution and the FAST-PT code.

The probe selection, the dataset's binning and its data files wait for the
$\chi^2$ section. The `int(...)` conversions are needed because several
bindings accept only a Python integer, not a float or a NumPy integer.

In [ ]:
# Init Cosmolike
# relativeFileName(key) returns the file named under key in the dataset
# file, with the dataset's folder prepended
ini = IniFile(os.path.normpath(os.path.join(path, data_file)))

lens_file = ini.relativeFileName('nz_lens_file')
source_file = ini.relativeFileName('nz_source_file')
lens_ntomo = ini.int("lens_ntomo")
source_ntomo = ini.int("source_ntomo")

# Reset every CosmoLike setting to its default value
ci.initial_setup()
ci.set_omp_threads(n = threads)

ci.init_photoz_conventions(
    interpolation_type = int(photoz_interpolation_type),
    zmid_convention = int(photoz_zmid_convention))

# set before the first init_accuracy_boost, as the likelihood does:
# init_accuracy_boost scales this fraction from the value it finds at
# its first call in the process
ci.init_fpt_internal_boost(internal_boost = float(internal_accuracyboost))

# The likelihood's lens-probe choices; cosmic shear does not read them
ci.init_nonlimber_accuracy_boost(nonlimber_boost = float(nonlimber_accuracyboost))
ci.init_adopt_limber_gs(adopt_limber_gs = int(adopt_limber_gs))
ci.init_adopt_limber_gg(adopt_limber_gg = int(adopt_limber_gg))
ci.init_include_HOD_GX(include_HOD_GX = 0)
ci.init_include_halo_IA(include_halo_IA = 0)

# Baseline table sizes and quadrature level; every wrapper call sets its own
ci.init_ntable_lmax(lmax = int(lmax))
ci.init_accuracy_boost(accuracy_boost = 1.0,
                       integration_accuracy = int(integration_accuracy))

# False: use the nonlinear matter power spectrum
ci.init_cosmo_runmode(is_linear = False)

# int(): these bindings refuse floats and NumPy integers
ci.init_redshift_distributions_from_files(
      lens_multihisto_file = lens_file,
      lens_ntomo = int(lens_ntomo),
      source_multihisto_file = source_file,
      source_ntomo = int(source_ntomo))

ci.init_IA(ia_model = int(IA_model),
           ia_redshift_evolution = int(IA_redshift_evolution),
           ia_code = int(IA_code))

## How $\Omega_m$ and $n_s$ change $C_\ell^{EE}$

The first sweep varies $\Omega_m$ from 0.20 to 0.38 in steps of 0.02, everything
else fixed, and draws the absolute spectra colored by $\Omega_m$; the cell also
prints the largest $|C_\ell^{BB}|$ over the sweep, zero for NLA. The second
varies $n_s$ from 0.85 to 1.09 and draws each spectrum relative to the fiducial
one, $C_\ell/C_\ell^{\rm fid}-1$. Both use 74 multipoles from 25 to 1485. A larger
$\Omega_m$ raises the lensing signal at every multipole; $n_s$ tilts the
primordial spectrum about $k=0.05$/Mpc, so its ratios change steadily with
multipole.

In [ ]:
# 74 multipoles from 25 to 1485; float endpoints make np.arange return
# floats, the type the bindings read as multipoles
ell = np.arange(25., 1500., 20.)

param = np.arange(0.2, 0.4, 0.02)

C_ss = []
largest_BB = 0.0
for x in param:
    # Cl: the E-mode spectra; C_BB: the B-mode spectra, zero for NLA
    (Cl, C_BB) = C_ss_tomo_limber(ell = ell, omegam = x)
    C_ss.append(Cl)
    largest_BB = max(largest_BB, np.max(np.abs(C_BB)))
print("Largest |C_ell^BB| over the Omega_m sweep (zero for NLA):", largest_BB)

In [ ]:
cnu.plot_C_ss_tomo_limber(ell = ell,
                          C_ss = C_ss,
                          param = param,
                          colorbarlabel = r"$\Omega_m$",
                          yaxislabelsize = 17,
                          yaxisticklabelsize = 14,
                          xaxisticklabelsize = 20)

In [ ]:
# 74 multipoles from 25 to 1485, as above
ell = np.arange(25., 1500., 20.)

param = np.arange(0.85, 1.1, 0.02)

C_ss = []
for x in param:
    # tmp: the B-mode spectra, not plotted
    (Cl, tmp) = C_ss_tomo_limber(ell = ell, ns = x)
    C_ss.append(Cl)

# The fiducial spectrum, the reference of the ratios
(ref, tmp) = C_ss_tomo_limber(ell = ell)

In [ ]:
# Reference mode: C/C_ref - 1; ylim (0.7, 1.35) is the band -30% to +35%
cnu.plot_C_ss_tomo_limber(ell = ell,
                          C_ss = C_ss,
                          C_ss_ref = ref,
                          param = param,
                          colorbarlabel = r"$n_s$",
                          ylim = (0.7, 1.35),
                          yaxislabelsize = 17,
                          yaxisticklabelsize = 14,
                          xaxisticklabelsize = 20)

## How $\Omega_m$ and $A_s$ change $\xi_\pm(\theta)$

The first sweep varies $\Omega_m$ from 0.20 to 0.40; each entry of `xi_theta` is
the tuple `(theta, xi_plus, xi_minus)` returned by `xi`. `plot_xi` draws $\xi_+$
for `pm = 1` and $\xi_-$ for `pm = -1`, as $\theta\,\xi\times10^4$ with $\theta$
in arcmin, on a linear axis, over the full angular range of the data vector.

In the flat-sky limit the $\xi_+$ weight is the Bessel function $J_0(\ell\theta)$
and the $\xi_-$ weight is $J_4(\ell\theta)$, which peaks at larger $\ell\theta$. At
a given angle $\xi_-$ therefore responds to higher multipoles, that is, to
smaller physical scales, where baryonic feedback and nonlinear modeling matter
most. This is why the DES Y6 scale cuts keep fewer $\xi_-$ entries (58 of 260)
than $\xi_+$ entries (152 of 260).

The last sweep evaluates $10^9A_s$ = 1.9, 2.1 and 2.3, the fiducial and $\pm0.2$
around it, and draws $\xi_+$ relative to the fiducial. The power spectrum is
proportional to $A_s$ in linear theory, so the large-angle ratios sit near
$1.9/2.1-1$ and $2.3/2.1-1$, about −9.5% and +9.5%; at small angles the
nonlinear power responds more strongly.

In [ ]:
param = np.arange(0.2, 0.41, 0.02)

xi_theta = []
for x in param:
    # Each entry is the tuple (theta, xi_plus, xi_minus)
    xi_theta.append(xi(omegam = x))

In [ ]:
# pm = 1 draws xi_+; thetashow is the horizontal range in arcmin
cnu.plot_xi(pm = 1,
            xi = xi_theta,
            param = param,
            colorbarlabel = r"$\Omega_m$",
            thetashow = [theta_min_arcmin, theta_max_arcmin],
            yaxislabelsize = 17,
            yaxisticklabelsize = 14,
            xaxisticklabelsize = 20)

In [ ]:
# pm = -1 draws xi_-
cnu.plot_xi(pm = -1,
            xi = xi_theta,
            param = param,
            colorbarlabel = r"$\Omega_m$",
            thetashow = [theta_min_arcmin, theta_max_arcmin],
            yaxislabelsize = 17,
            yaxisticklabelsize = 14,
            xaxisticklabelsize = 20)

In [ ]:
# 10^9 A_s = 1.9, 2.1 and 2.3; 2.1 is the YAML fiducial, so the middle
# curve coincides with the reference
param = np.array([1.9, 2.1, 2.3])
xi_theta = []
for x in param:
    xi_theta.append(xi(As_1e9 = x))
xi_theta_ref = xi()

In [ ]:
# Reference mode: xi_+/xi_+,fid - 1; ylim (0.86, 1.14) is the band -14% to
# +14%
cnu.plot_xi(pm = 1,
            xi = xi_theta,
            xi_ref = xi_theta_ref,
            param = param,
            colorbarlabel = r"$10^9 A_s$",
            ylim = (0.86, 1.14),
            thetashow = [theta_min_arcmin, theta_max_arcmin],
            yaxislabelsize = 17,
            yaxisticklabelsize = 14,
            xaxisticklabelsize = 20)

## Changing the angular binning without restarting the kernel

The *kernel* is the Python process that runs the cells. CosmoLike keeps
expensive intermediate results, such as the bin-averaged kernels that turn
$C_\ell$ into $\xi_\pm(\theta)$, in static arrays: C memory that survives between
calls. Each cached table carries a key; `ci.init_binning` assigns a new key
whenever the number of bins or the angular range changes, so the next call
rebuilds the kernels instead of reusing those of the previous binning.

The cell computes $\xi_\pm$ with 9, 12, 17 and 44 bins between 2.5 and 995.27
arcmin, and the figures mark each binning with its own symbol. The points of all
binnings should follow one smooth curve, wide bins deviating slightly because
each point averages over its bin; a stale cache would instead place the
previous binning's values at the new angles.

In [ ]:
nthetas = [9, 12, 17, 44]

result = []
for x in nthetas:
    # A new binning on every call: CosmoLike rebuilds its angular kernels
    result.append(xi(ntheta = x))

In [ ]:
cnu.plot_xi(pm = 1,
            xi = result,
            marker = ['s', 'o', ">", "<"],
            thetashow = [theta_min_arcmin, theta_max_arcmin])

In [ ]:
cnu.plot_xi(pm = -1,
            xi = result,
            marker = ['s', 'o', ">", "<"],
            thetashow = [theta_min_arcmin, theta_max_arcmin])

## Baryonic feedback from tabulated simulations in $C_\ell^{EE}$ and $\xi_\pm(\theta)$

Gas physics (cooling, star formation, energy injected by supernovae and active
galactic nuclei) moves matter inside and around halos and changes the matter
power spectrum at small scales. `ci.init_baryons_contamination(sim, allsims)`
multiplies CosmoLike's nonlinear matter power by the ratio of the power spectrum
of one hydrodynamical simulation to that of its dark-matter-only counterpart,
tabulated in the HDF5 file (a binary table format) `baryons_logPkR.h5` named in
the dataset. The ratio is a fixed table: it does not change with cosmology or
with any feedback parameter.

| Label | Simulation |
| --- | --- |
| `TNG100-1` | IllustrisTNG, TNG100 box |
| `HzAGN-1` | Horizon-AGN |
| `mb2-1` | MassiveBlack-II |
| `illustris-1` | Illustris |
| `eagle-1` | EAGLE |
| `owls_AGN-1` | cosmo-OWLS AGN, heating temperature $10^{8.0}$ K |
| `owls_AGN-2` | cosmo-OWLS AGN, heating temperature $10^{8.5}$ K |
| `owls_AGN-3` | cosmo-OWLS AGN, heating temperature $10^{8.7}$ K |
| `BAHAMAS-2` | BAHAMAS, heating temperature $10^{7.6}$ K |

The cells compute these nine scenarios and the reference without feedback, all
on the gravity-only Takahashi spectrum (`halofit_version = "takahashi"`), then
draw each scenario relative to the reference ($C/C^{\rm ref}-1$ and
$\xi/\xi^{\rm ref}-1$), named by a legend instead of a colorbar. The real-space
curves use 40 angular bins, for smoother lines. EXAMPLE_EVALUATE1.ipynb ends with
the parametric emulators of the `bfmt` theory block, the other route to the same
ratio.

In [ ]:
# 750 multipoles from 3 to 7493 (float endpoints: np.arange returns floats)
ell = np.arange(3., 7500., 10.)
# Simulation names in the HDF5 table baryons_logPkR.h5
param = ("TNG100-1", "HzAGN-1", "mb2-1", "illustris-1", "eagle-1",
         "owls_AGN-1", "owls_AGN-2", "owls_AGN-3", "BAHAMAS-2")

# A simulation ratio multiplies a gravity-only spectrum, and the reference
# uses the same one: every call passes the Takahashi Halofit
C_ss = []
for x in param:
    (Cl, tmp) = C_ss_tomo_limber(ell = ell, baryon_sims = x,
                                 halofit_version = "takahashi")
    C_ss.append(Cl)
(C_ss_ref, tmp) = C_ss_tomo_limber(ell = ell, halofit_version = "takahashi")

xi_theta = []
for x in param:
    # 40 angular bins draw smoother curves than the 26 of the data vector
    xi_theta.append(xi(baryon_sims = x, ntheta = 40,
                       halofit_version = "takahashi"))
xi_theta_ref = xi(ntheta = 40, halofit_version = "takahashi")

In [ ]:
# ylim (0.6, 1.3): the band -40% to +30% of C/C_ref - 1. tab10 gives nine
# distinct dark colors; the linestyles separate curves of similar color.
cnu.plot_C_ss_tomo_limber(ell = ell,
                          C_ss = C_ss,
                          C_ss_ref = C_ss_ref,
                          lmin = 10,
                          lmax = 5000,
                          cmap = "tab10",
                          ylim = (0.6, 1.3),
                          legend = param,
                          legendfontsize = 18,
                          linewidth = [2.0],
                          linestyle = ['solid', 'dashed', 'dashdot', 'dotted'])

In [ ]:
# legend names the curves instead of a colorbar; ylim (0.81, 1.09) is the
# band -19% to +9% of xi/xi_ref - 1
cnu.plot_xi(pm = 1,
            xi = xi_theta,
            xi_ref = xi_theta_ref,
            cmap = "tab10",
            ylim = (0.81, 1.09),
            linewidth = [2.0],
            linestyle = ['solid', 'dashed', 'dashdot', 'dotted'],
            legend = param,
            legendfontsize = 18,
            thetashow = [theta_min_arcmin, theta_max_arcmin],
            bintextpos = [[0.2, 0.875], [0.2, 0.875]])

In [ ]:
cnu.plot_xi(pm = -1,
            xi = xi_theta,
            xi_ref = xi_theta_ref,
            cmap = "tab10",
            ylim = (0.81, 1.09),
            linewidth = [2.0],
            linestyle = ['solid', 'dashed', 'dashdot', 'dotted'],
            legend = param,
            legendfontsize = 18,
            thetashow = [theta_min_arcmin, theta_max_arcmin],
            bintextpos = [[0.2, 0.875], [0.2, 0.875]])

## Accuracy settings: how `AccuracyBoost` changes $C_\ell^{EE}$ and $\xi_+(\theta)$

`AccuracyBoost` refines three things together: CAMB runs with accuracy
`CAMBAccuracyBoost`$\times(1+(\texttt{AccuracyBoost}-1)/3)$, the CosmoLike tables
and the CAMB grids grow with `AccuracyBoost`, and the wrapper raises the
quadrature level by $|3(\texttt{AccuracyBoost}-1)|$, rounded down, so a boost
below 1 raises it too.

The spectra use `AccuracyBoost` from 0.45 to 1.95 in steps of 0.25 and the
$\xi_+$ curves 0.50 to 1.75, each drawn relative to `AccuracyBoost = 1`, on bands
of ±4% and ±1%. The $\chi^2$ sections below express such
differences in units of the forecast errors.

In [ ]:
# 283 multipoles from 10 to 8470 (float endpoints: np.arange returns floats)
ell = np.arange(10., 8500., 30.)

# AccuracyBoost 0.45 to 1.95 refines CAMB, the CosmoLike tables and the
# quadrature level together
param = np.arange(0.45, 2.0, 0.25)

C_ss = []
for x in param:
    (Cl, tmp) = C_ss_tomo_limber(ell = ell, AccuracyBoost = x)
    C_ss.append(Cl)

# The AccuracyBoost = 1 spectrum, the reference of the ratios
(C_ss_ref, tmp) = C_ss_tomo_limber(ell = ell)

In [ ]:
# ylim (0.96, 1.04): the band -4% to +4% of C/C_ref - 1
cnu.plot_C_ss_tomo_limber(ell = ell,
                          C_ss = C_ss,
                          C_ss_ref = C_ss_ref,
                          param = param,
                          lmin = 10,
                          lmax = 8000,
                          ylim = (0.96, 1.04),
                          colorbarlabel = "Accuracy Boost")

In [ ]:
# AccuracyBoost 0.50 to 1.75, 40 angular bins for smoother curves
param = np.arange(0.5, 2.0, 0.25)
xi_theta_ref = xi(ntheta = 40)

xi_theta = []
for x in param:
    xi_theta.append(xi(ntheta = 40, AccuracyBoost = x))

In [ ]:
# ylim (0.99, 1.01): the band -1% to +1% of xi_+/xi_+ref - 1
cnu.plot_xi(pm = 1,
            xi = xi_theta,
            xi_ref = xi_theta_ref,
            param = param,
            colorbarlabel = "Accuracy Boost",
            thetashow = [theta_min_arcmin, theta_max_arcmin],
            ylim = (0.99, 1.01))

## $\chi^2$ against a synthetic DES Y6 data vector

`data/DESY6.dataset` names an all-ones data vector and an identity covariance,
so a $\chi^2$ against them measures nothing. This section installs in `ci` the
three inputs of a survey $\chi^2$, built from the project's own files:

| Input | Content |
| --- | --- |
| Covariance | The `total` array of `covariance/forecast_real.npz`: the Gaussian, super-sample and connected non-Gaussian forecast of EXAMPLE_EVALUATE_COVARIANCE.ipynb, in the 1,300-entry order of the data vector. |
| Mask | `data/DESY6.mask`, the DES Y6 scale cuts: 541 of the 1,300 entries, 210 of them cosmic-shear entries, 152 of $\xi_+$ and 58 of $\xi_-$. |
| Data vector | Synthetic: this notebook's fiducial prediction with the run settings, not a measurement. |

`init_data_real` reads files, so the first cell writes the covariance as text
into a temporary folder, which Python deletes when the kernel stops. The file
holds the 1,300 variances and the upper triangle of the block the scale cuts
keep: the reader fills each mirror entry itself and zeroes every off-diagonal
entry of a masked row. The cell defines two functions:

- `write_data_vector(datavector, filename)` writes a data vector in the
  two-column layout `init_data_real` reads;
- `initialize_with_data(cov_file, mask_file, data_vector_file)` repeats the
  whole initialization from `initial_setup`, in the order of the likelihood's
  `initialize`, now with the probe selection, the 26 angular bins of the
  dataset and `init_data_real`. The reset is needed because `init_data_real`
  adds the data-vector length to the length CosmoLike holds: installing a
  second data vector without it would double that length.

Here `CLprobe = "xi"` selects cosmic shear: `init_probes` sets the mask of
every other entry to zero, so 210 entries enter $\chi^2$, 152 of $\xi_+$ and 58 of
$\xi_-$. The inverse covariance then belongs to the shear block alone: the
reader zeroes every covariance entry that couples a shear entry to a masked
galaxy–galaxy lensing or clustering entry before inverting, so the $\chi^2$ is
the cosmic-shear $\chi^2$, not the shear corner of the 3×2pt inverse.

The second cell defines `get_datavector(...)`, which runs the stages of the
wrappers with every nuisance parameter the likelihood sends and returns
`compute_data_vector_masked()`: all 1,300 entries, masked ones and every
non-shear entry set to zero, with the shear calibration $(1+m_i)(1+m_j)$ on
$\xi_\pm$.
`ci.compute_chi2(datavector = t)` then returns
$\chi^2=(d-t)^{\mathsf T}C^{-1}(d-t)$ over the retained entries, with $d$ the
installed data vector, $t$ the prediction and $C^{-1}$ the inverse of the
retained block of the covariance.

The third cell computes the fiducial prediction under the dataset's placeholder
files, whose mask lets `compute_data_vector_masked` run, and installs it as the
synthetic data vector, so its $\chi^2$ is zero by construction. It then prints
the $\chi^2$ of HMcode 2020 without its feedback term
(`halofit_version = "mead2020"`) against it: the size of the YAML's feedback
term in units of the forecast errors, after the scale cuts.

In [ ]:
# tempfile creates the temporary folder of the text files below
import tempfile

# Files of the dataset; ini also names its placeholder data files
ini = IniFile(os.path.normpath(os.path.join(path, data_file)))
scale_cut_file = os.path.join(path, "DESY6.mask")

# The forecast covariance [1300, 1300], in the order of the data vector
forecast_file = os.path.join(os.environ['ROOTDIR'], "projects", "des_y6",
                             "covariance", "forecast_real.npz")
if not os.path.exists(forecast_file):
    raise FileNotFoundError(
        f"{forecast_file} does not exist: the chi^2 sections need the "
        "forecast covariance; run EXAMPLE_EVALUATE_COVARIANCE.ipynb or "
        "covariance/compute_covariance.py first (README, computing covariances)")
# allow_pickle=False reads numbers only; the with block closes the archive
with np.load(forecast_file, allow_pickle = False) as archive:
    forecast_covariance = np.array(archive["total"])

# DESY6.mask: column 0 the entry index 0..1299, column 1 = 1 to keep it
scale_cut_table = np.loadtxt(scale_cut_file)
ndata = scale_cut_table.shape[0]
if forecast_covariance.shape != (ndata, ndata):
    raise ValueError(
        f"forecast covariance shape {forecast_covariance.shape}, scale cuts "
        f"for {ndata} entries: they must agree; recompute the forecast with "
        "EXAMPLE_EVALUATE_COVARIANCE.ipynb for DESY6.dataset")
if not np.array_equal(scale_cut_table[:, 0], np.arange(ndata)):
    raise ValueError(
        f"DESY6.mask does not list the entries 0..{ndata - 1} in order: "
        "restore data/DESY6.mask from the project repository")

# The text layout init_data_real reads: one line "i j C_ij" per stored
# entry. The reader fills the mirror entry (j, i) itself and zeroes every
# off-diagonal entry of a masked row, so the file holds the 1,300
# variances and the upper triangle of the block the scale cuts keep.
cut_entries = np.flatnonzero(scale_cut_table[:, 1] == 1)
masked_entries = np.flatnonzero(scale_cut_table[:, 1] == 0)
# np.triu_indices(n) lists the index pairs (a, b), a <= b, of an n x n block
(block_row, block_column) = np.triu_indices(cut_entries.size)
row_index = np.concatenate([cut_entries[block_row], masked_entries])
column_index = np.concatenate([cut_entries[block_column], masked_entries])
covariance_entries = forecast_covariance[row_index, column_index]

# A folder that lives while the kernel keeps `scratch`; Python deletes it
# when the kernel stops
scratch = tempfile.TemporaryDirectory(prefix = "des_y6_chi2_")
covariance_file = os.path.join(scratch.name, "forecast_total.cov")
# %.17e keeps every digit of a double, so the reader gets the same numbers
np.savetxt(covariance_file,
           np.column_stack([row_index, column_index, covariance_entries]),
           fmt = "%d %d %.17e")


def write_data_vector(datavector, filename):
    """Write a data vector as init_data_real reads it: lines "i d_i".

    Arguments:
      datavector = 1,300 entries in the order of compute_data_vector_masked
      filename   = file written (replaced when it exists)
    """
    entry_index = np.arange(len(datavector))
    np.savetxt(filename, np.column_stack([entry_index, datavector]),
               fmt = "%d %.17e")


def initialize_with_data(cov_file, mask_file, data_vector_file):
    """Run the likelihood's initialization with three data files.

    initial_setup comes first: init_data_real adds the data-vector length
    to the length CosmoLike holds, so a second call without the reset
    would double it. The sequence is the one of the initialization cell,
    plus the probe selection, the dataset binning and init_data_real, in
    the order of the likelihood's initialize.

    Arguments:
      cov_file, mask_file, data_vector_file = text files in the layouts
                 init_data_real reads, 1,300 entries each

    Side effects:
      replaces every CosmoLike setting; ci.compute_chi2, ci.get_mask and
      ci.get_inv_cov_masked then use the three files.
    """
    ci.initial_setup()
    ci.set_omp_threads(n = threads)
    ci.init_probes(possible_probes = CLprobe)
    ci.init_binning(ntheta_bins = int(ini.int("n_theta")),
                    theta_min_arcmin = ini.float("theta_min_arcmin"),
                    theta_max_arcmin = ini.float("theta_max_arcmin"))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))
    # initial_setup resets the FAST-PT grid fraction to the C default; set
    # it again before the first init_accuracy_boost
    ci.init_fpt_internal_boost(internal_boost = float(internal_accuracyboost))
    ci.init_nonlimber_accuracy_boost(nonlimber_boost = float(nonlimber_accuracyboost))
    ci.init_adopt_limber_gs(adopt_limber_gs = int(adopt_limber_gs))
    ci.init_adopt_limber_gg(adopt_limber_gg = int(adopt_limber_gg))
    ci.init_include_HOD_GX(include_HOD_GX = 0)
    ci.init_include_halo_IA(include_halo_IA = 0)
    ci.init_ntable_lmax(lmax = int(lmax))
    ci.init_accuracy_boost(accuracy_boost = 1.0,
                           integration_accuracy = int(integration_accuracy))
    ci.init_cosmo_runmode(is_linear = False)
    ci.init_redshift_distributions_from_files(
          lens_multihisto_file = ini.relativeFileName('nz_lens_file'),
          lens_ntomo = int(ini.int("lens_ntomo")),
          source_multihisto_file = ini.relativeFileName('nz_source_file'),
          source_ntomo = int(ini.int("source_ntomo")))
    ci.init_data_real(COV = cov_file, MASK = mask_file, DATA = data_vector_file)
    ci.init_IA(ia_model = int(IA_model),
               ia_redshift_evolution = int(IA_redshift_evolution),
               ia_code = int(IA_code))

In [ ]:
def get_datavector(omegam = omegam,
                   omegab = omegab,
                   H0 = H0,
                   ns = ns,
                   As_1e9 = As_1e9,
                   w = w,
                   w0pwa = w0pwa,
                   A1 = [DES_A1_1, DES_A1_2, 0, 0],
                   A2 = [DES_A2_1, DES_A2_2, 0, 0],
                   BTA = [DES_BTA_1, 0, 0, 0],
                   shear_photoz_bias = [DES_DZ_S1, DES_DZ_S2, DES_DZ_S3, DES_DZ_S4],
                   M = [DES_M1, DES_M2, DES_M3, DES_M4],
                   halofit_version = halofit_version,
                   AccuracyBoost = 1.0,
                   kmax = kmax,
                   k_per_logint = k_per_logint,
                   CAMBAccuracyBoost = CAMBAccuracyBoost,
                   CLAccuracyBoost = 1.0,
                   CLIntegrationAccuracy = integration_accuracy):
    """Return the masked model vector [1300] at one parameter point.

    Runs the stages of the wrappers with every source-side nuisance parameter
    the likelihood sends, then compute_data_vector_masked: masked entries
    are zero, and the shear calibration is applied.
    The binning is reset to the dataset's 26 bins, which the real-space
    wrappers may have changed.
    """
    # Stage 1, CAMB
    (log10k_interp_2D, z_interp_2D, lnPL, lnPNL, G_growth, z_growth,
     z_interp_1D, chi, omegan2, lnPL_cb) = cnu.get_camb_cosmology(
        omegam = omegam, omegab = omegab, H0 = H0, ns = ns,
        As_1e9 = As_1e9, w = w, w0pwa = w0pwa, mnu = mnu,
        non_linear_emul = non_linear_emul,
        halofit_version = halofit_version,
        AccuracyBoost = AccuracyBoost, kmax = kmax,
        k_per_logint = k_per_logint,
        CAMBAccuracyBoost = CAMBAccuracyBoost,
        CLAccuracyBoost = CLAccuracyBoost)

    # Stage 2, accuracy of the C code, as in the wrappers
    ci.set_omp_threads(n = threads)
    CLAccuracyBoost = CLAccuracyBoost * AccuracyBoost
    CLIntegrationAccuracy = max(0, CLIntegrationAccuracy + abs(3*(CLAccuracyBoost-1.0)))
    ci.init_ntable_lmax(lmax = int(lmax + 20000*(CLAccuracyBoost-1)))
    ci.init_accuracy_boost(accuracy_boost = CLAccuracyBoost,
                           integration_accuracy = int(CLIntegrationAccuracy))
    ci.init_photoz_conventions(
        interpolation_type = int(photoz_interpolation_type),
        zmid_convention = int(photoz_zmid_convention))
    ci.init_binning(ntheta_bins = int(ntheta),
                    theta_min_arcmin = theta_min_arcmin,
                    theta_max_arcmin = theta_max_arcmin)

    # Stage 3: copy the CAMB tables into CosmoLike
    ci.set_cosmology(omegam = omegam,
                     omegab = omegab,
                     H0 = H0,
                     log10k_2D = log10k_interp_2D,
                     z_2D = z_interp_2D,
                     lnP_linear = lnPL,
                     lnP_linear_cb = lnPL_cb,
                     lnP_nonlinear = lnPNL,
                     G = G_growth,
                     z_G = z_growth,
                     z_1D = z_interp_1D,
                     chi = chi,
                     omegan2 = omegan2)

    # Stage 4: the source-side nuisance parameters, as the likelihood's
    # set_source_related sends them
    ci.set_nuisance_shear_calib(M = M)
    ci.set_nuisance_shear_photoz(bias = shear_photoz_bias)
    ci.set_nuisance_ia(A1 = A1, A2 = A2, B_TA = BTA)
    # No tabulated feedback in the data vector
    ci.reset_bary_struct()

    # Stage 5: all 1,300 entries, masked ones set to zero
    return np.array(ci.compute_data_vector_masked())

In [ ]:
# The dataset's placeholder files (identity covariance, all-ones data,
# ones.mask) only let compute_data_vector_masked run for the first
# prediction: the synthetic data vector
initialize_with_data(cov_file = ini.relativeFileName('cov_file'),
                     mask_file = ini.relativeFileName('mask_file'),
                     data_vector_file = ini.relativeFileName('data_file'))
dv_fiducial = get_datavector()

# The forecast covariance, the scale cuts and the fiducial prediction as
# the synthetic data vector
fiducial_data_file = os.path.join(scratch.name, "fiducial.modelvector")
write_data_vector(datavector = dv_fiducial, filename = fiducial_data_file)
initialize_with_data(cov_file = covariance_file,
                     mask_file = scale_cut_file,
                     data_vector_file = fiducial_data_file)

# get_mask returns the 0/1 flag of every data-vector entry: their sum is
# the number of entries that enter chi^2
n_retained = int(np.sum(ci.get_mask()))
chi2_fiducial = ci.compute_chi2(datavector = get_datavector())
print("chi2 of the fiducial prediction (halofit_version = %s) = %.4f "
      "over %d data points" % (halofit_version, chi2_fiducial, n_retained))
# HMcode 2020 without its feedback term, against the same data vector
dv_no_feedback = get_datavector(halofit_version = "mead2020")
chi2_no_feedback = ci.compute_chi2(datavector = dv_no_feedback)
print("chi2 of HMcode 2020 without feedback (mead2020) = %.4f "
      "over %d data points" % (chi2_no_feedback, n_retained))

## Quadrature check: $\Delta\chi^2$ against the integration level

The next cell evaluates $\chi^2$ at quadrature levels 0 to 9, everything else
fixed, and subtracts the level-0 value; the plot after it shows
$\Delta\chi^2=\chi^2(\text{level})-\chi^2(0)$ against the level. Level 0 is the
setting of the synthetic data vector, so $\chi^2(0)=0$ and each $\Delta\chi^2$ is
the $\chi^2$ of the shift between that level's prediction and level 0. Each level
selects a fixed Gauss–Legendre rule, a set of points and weights for the
line-of-sight integrals: for the shear spectra 96, 128,
256, 512 and 1024 points at levels 0 to 4. Higher levels reuse the 1024-point
rule, so the curve is flat beyond level 4.

The frozen-reference tests of the Cocoa projects that ship a survey covariance
accept $\Delta\chi^2<0.2$. The DES Y6 tests compare data-vector entries instead,
because the shipped covariance is the identity: run
`python -m pytest ./projects/des_y6/tests/data_vector` from `cocoa/Cocoa` for
the accuracy, cache-consistency and frozen-reference checks.

In [ ]:
param = np.arange(0, 10, 1)
chi2 = []
for x in param:
    datavector = get_datavector(CLIntegrationAccuracy = x)
    chi2.append(ci.compute_chi2(datavector = datavector))
chi2 = np.array(chi2)
# Delta chi^2 relative to quadrature level 0, the setting of the synthetic
# data vector (chi2[0] = 0)
chi2 = chi2 - chi2[0]

In [ ]:
# Delta chi^2 of each quadrature level against level 0
plt.plot(param, chi2, marker = 'o')
plt.xlabel("quadrature level (CLIntegrationAccuracy)")
plt.ylabel(r"$\Delta\chi^2$ against level 0")
plt.show()

## Interpolation check: $\Delta\chi^2$ against `CLAccuracyBoost`

The next cell evaluates $\chi^2$ at `CLAccuracyBoost` = 1, 2, ..., 9 and subtracts
the boost-1 value, zero for the same reason. This boost multiplies the sizes of
CosmoLike's interpolation tables and the redshift and wavenumber grids of the
CAMB tables, and raises the multipole cutoff of `init_ntable_lmax` by 20,000 per
unit; CAMB's own accuracy stays fixed. `get_datavector` also raises the
quadrature level by $3(\text{boost}-1)$ above the default level 0, so this scan
refines interpolation and quadrature together; from boost 3 on, the quadrature
uses its largest rule. The highest boosts build the largest tables and take the
longest.

In [ ]:
param = np.arange(1, 10, 1)
chi2 = []
for x in param:
    datavector = get_datavector(CLAccuracyBoost = x)
    chi2.append(ci.compute_chi2(datavector = datavector))
chi2 = np.array(chi2)
# Delta chi^2 relative to CLAccuracyBoost = 1, the setting of the synthetic
# data vector (chi2[0] = 0); the quadrature level is 3 (boost - 1), so it
# rises with the boost
chi2 = chi2 - chi2[0]

In [ ]:
# Delta chi^2 of each CLAccuracyBoost against boost 1
plt.plot(param, chi2, marker = 'o')
plt.xlabel("CLAccuracyBoost")
plt.ylabel(r"$\Delta\chi^2$ against boost 1")
plt.show()